In [0]:
%pip install scikit-learn mlflow azure-storage-blob

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 121.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 116.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 621.4/621.4 kB 45.5 MB/s eta 0:00:00
  Attempting uninstall: wcwidth
    Found existing installation: wcwidth 0.2.5
    Not uninstalling wcwidth at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4a5dd00a-52c0-4a5a-81fe-beba701b4d83
    Can't uninstall 'wcwidth'. No files were found to uninstall.
  Attempting uninstall: blinker
    Found existing installation: blinker 1.7.0
    Not uninstalling blinker at /usr/lib/python3/dist-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-4a5dd00a-52c0-4a5a-81fe-beba701b4d83
    Can't uninstall 'blinker'. No file

In [0]:
dbutils.library.restartPython()

In [0]:
from azure.storage.blob import BlobServiceClient
import pandas as pd
import numpy as np
import io
import mlflow
import mlflow.sklearn
from sklearn.ensemble import IsolationForest
from pyspark.sql import functions as F

storage_account_name = "financecomplaintssa"
storage_account_key = dbutils.secrets.get(scope="dbScope", key="StorageAccountKey")
connection_string = (
    f"DefaultEndpointsProtocol=https;AccountName={storage_account_name};"
    f"AccountKey={storage_account_key};EndpointSuffix=core.windows.net"
)
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

def write_df_to_container(df, container_name, filename):
    pdf = df.toPandas() if hasattr(df, "toPandas") else df
    csv_buffer = io.StringIO()
    pdf.to_csv(csv_buffer, index=False)
    container_client = blob_service_client.get_container_client(container_name)
    blob_client = container_client.get_blob_client(filename)
    blob_client.upload_blob(csv_buffer.getvalue(), overwrite=True)
    print(f"Uploaded {filename} to {container_name}/ - {len(pdf)} rows")

## 1. Load Silver tables and rebuild the Category join

In [0]:
df_complaints = spark.read.table("college_project.default.consumercomplaints_silver")
df_products   = spark.read.table("college_project.default.productcategoryreference_silver")




df_joined = df_complaints.join(F.broadcast(df_products), on="Product", how="left")
print("Complaint rows:", df_complaints.count())
print("Joined rows   :", df_joined.count())


Complaint rows: 62516
Joined rows   : 62516


## 2. Feature engineering for the anomaly model

In [0]:
df_features = (
    df_joined
    .withColumn("Days_to_Respond", F.datediff(F.col("Date_received"), F.col("Date_submitted")))
    .withColumn("is_timely", F.when(F.col("Timely_Response") == "Yes", 1).otherwise(0))
    .select(
        "Complaint_ID", "Category", "Sub_Category", "Submitted_Via",
        "Company_Response", "Days_to_Respond", "is_timely"
    ))

pdf = df_features.toPandas()

# Label-encode the low-cardinality categoricals; keep raw values for reporting
for col in ["Category", "Sub_Category", "Submitted_Via", "Company_Response"]:
    pdf[col] = pdf[col].fillna("Unknown")
    pdf[col + "_code"] = pdf[col].astype("category").cat.codes

pdf["Days_to_Respond"] = pdf["Days_to_Respond"].fillna(pdf["Days_to_Respond"].median())

feature_cols = [
    "Days_to_Respond", "is_timely",
    "Category_code", "Sub_Category_code",
    "Submitted_Via_code", "Company_Response_code",
]

X = pdf[feature_cols]
X.describe()

,Days_to_Respond,is_timely,Category_code,Sub_Category_code,Submitted_Via_code,Company_Response_code
count,62516.000000,62516.000000,62516.000000,62516.000000,62516.000000,62516.000000
mean,1.224886,0.937664,1.912694,2.116082,4.547220,1.475350
std,5.649298,0.241767,1.732866,2.046663,0.887548,0.749491
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,1.000000,0.000000,0.000000,4.000000,1.000000
50%,0.000000,1.000000,2.000000,2.000000,5.000000,1.000000
75%,0.000000,1.000000,4.000000,2.000000,5.000000,2.000000
max,275.000000,1.000000,4.000000,8.000000,6.000000,4.000000


## 3. Train Isolation Forest, tracked with MLflow

In [0]:
mlflow.set_experiment("/Shared/finance_complaints_data_quality")
contamination = 0.05  # expected proportion of anomalous records - tune as needed

with mlflow.start_run(run_name="isolation_forest_data_quality") as run:
    model = IsolationForest(
        n_estimators=200,
        contamination=contamination,
        random_state=42,
    )
    model.fit(X)
    pdf["anomaly_score"] = model.decision_function(X)   # higher = more normal
    pdf["is_anomaly"] = (model.predict(X) == -1)
    anomaly_rate = float(pdf["is_anomaly"].mean())
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("contamination", contamination)
    mlflow.log_param("feature_cols", feature_cols)
    mlflow.log_metric("anomaly_rate", anomaly_rate)
    mlflow.sklearn.log_model(model, artifact_path="isolation_forest_model", skops_trusted_types=["sklearn.tree._tree.Tree"])
    run_id = run.info.run_id
    print(f"MLflow run ID: {run_id}")
    print(f"Flagged {int(pdf['is_anomaly'].sum())} anomalies out of {len(pdf)} rows ({anomaly_rate:.2%})")

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc
2026/09/17 05:44:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://adb-7405613100519355.15.azuredatabricks.net/ml/experiments/2882028168636383/models/m-ce17a2a38c354e929e07ff0ecde22bda?o=7405613100519355
2026/09/17 05:44:42 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.16.1/ml/model/signatures.html for instructions on setting signat

MLflow run ID: daea69e6b3844d1cabb6360cae3bb6ef
Flagged 3120 anomalies out of 62516 rows (4.99%)


## 4. Write anomaly detail to Gold (Delta table + ADLS CSV for Power BI)

In [0]:
df_anomalies = spark.createDataFrame(
    pdf[["Complaint_ID", "Category", "Sub_Category", "Company_Response",
         "Days_to_Respond", "anomaly_score", "is_anomaly"]])

df_anomalies.write.mode("overwrite").saveAsTable("college_project.default.complaints_anomaly_gold")
write_df_to_container(df_anomalies, "gold", "complaints_anomaly_gold.csv")
df_anomalies.filter(F.col("is_anomaly")).orderBy("anomaly_score").show(20, truncate=False)

Uploaded complaints_anomaly_gold.csv to gold/ - 62516 rows
+------------+-----------+-----------------------+---------------------------+---------------+--------------------+----------+
|Complaint_ID|Category   |Sub_Category           |Company_Response           |Days_to_Respond|anomaly_score       |is_anomaly|
+------------+-----------+-----------------------+---------------------------+---------------+--------------------+----------+
|7115264     |Collections|Third-Party Collections|In progress                |49             |-0.15155875577240874|true      |
|7197577     |Collections|Third-Party Collections|In progress                |35             |-0.14394815914464598|true      |
|7256518     |Collections|Third-Party Collections|In progress                |33             |-0.1406231415284852 |true      |
|6969984     |Payments   |Money Movement         |In progress                |56             |-0.13903059419515573|true      |
|7244155     |Banking    |Deposit Accounts       |In

## 5. Rule-based quality dimensions (Completeness, Validity, Uniqueness, Consistency)

In [0]:
total_rows = df_complaints.count()
key_cols = ["Complaint_ID", "Product", "Date_submitted", "Date_received",
            "Timely_Response", "Company_Response"]
non_null_row = df_complaints.select([
    F.sum(F.when(F.col(c).isNotNull(), 1).otherwise(0)).alias(c) for c in key_cols
]).collect()[0]
completeness_pct = round(
    sum(non_null_row[c] for c in key_cols) / (total_rows * len(key_cols)) * 100, 1)
valid_dates = df_joined.filter(
    F.col("Date_submitted").isNotNull() &
    F.col("Date_received").isNotNull() &
    (F.datediff(F.col("Date_received"), F.col("Date_submitted")) >= 0)).count()
valid_timely = df_complaints.filter(F.col("Timely_Response").isin("Yes", "No")).count()
validity_pct = round(((valid_dates + valid_timely) / (2 * total_rows)) * 100, 1)
unique_ids = df_complaints.select("Complaint_ID").distinct().count()
uniqueness_pct = round(unique_ids / total_rows * 100, 1)
matched_category = df_joined.filter(F.col("Category").isNotNull()).count()
consistency_pct = round(matched_category / total_rows * 100, 1)
ml_normality_pct = round((1 - anomaly_rate) * 100, 1)
print(f"Completeness : {completeness_pct}%")
print(f"Validity     : {validity_pct}%")
print(f"Uniqueness   : {uniqueness_pct}%")
print(f"Consistency  : {consistency_pct}%")
print(f"ML Normality : {ml_normality_pct}%")

Completeness : 99.6%
Validity     : 98.8%
Uniqueness   : 100.0%
Consistency  : 100.0%
ML Normality : 95.0%


## 6. Combine into the final weighted Data Quality Score

In [0]:
final_score = round(
    0.25 * completeness_pct +
    0.20 * validity_pct +
    0.20 * consistency_pct +
    0.15 * uniqueness_pct +
    0.20 * ml_normality_pct,
    1)

if final_score >= 90:
    status = "HIGH QUALITY"
elif final_score >= 75:
    status = "MODERATE QUALITY"
else:
    status = "LOW QUALITY"

print("=" * 40)
print("DATA QUALITY REPORT")
print("=" * 40)
print(f"Total Records       : {total_rows}")
print(f"Completeness        : {completeness_pct}%")
print(f"Validity            : {validity_pct}%")
print(f"Uniqueness          : {uniqueness_pct}%")
print(f"Consistency         : {consistency_pct}%")
print(f"ML Normality        : {ml_normality_pct}%")
print("-" * 40)
print(f"Overall Quality     : {final_score}%")
print(f"STATUS              : {status}")

report_pdf = pd.DataFrame([{
    "total_records": total_rows,
    "completeness_pct": completeness_pct,
    "validity_pct": validity_pct,
    "uniqueness_pct": uniqueness_pct,
    "consistency_pct": consistency_pct,
    "ml_normality_pct": ml_normality_pct,
    "overall_quality_score": final_score,
    "status": status,
    "mlflow_run_id": run_id,
}])

df_report = spark.createDataFrame(report_pdf)
df_report.write.mode("overwrite").saveAsTable("college_project.default.data_quality_report_gold")
write_df_to_container(df_report, "gold", "data_quality_report_gold.csv")

DATA QUALITY REPORT
Total Records       : 62516
Completeness        : 99.6%
Validity            : 98.8%
Uniqueness          : 100.0%
Consistency         : 100.0%
ML Normality        : 95.0%
----------------------------------------
Overall Quality     : 98.7%
STATUS              : HIGH QUALITY
Uploaded data_quality_report_gold.csv to gold/ - 1 rows
